In [ ]:
import os
import gradio as gr

# LangChain imports
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import Chroma
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.chains import RetrievalQA

# ctransformers + LangChain LLM wrapper
from langchain.llms import CTransformers

In [ ]:
# ------------------------
# Paths (change if needed)
# ------------------------
CHROMA_DIR = "./chroma_db_mistral"
EMBEDDING_MODEL_PATH = r"C:\Local LLM and Embedding Models\Embedding Model\Sentence Transformers_AllMiniLM"
MODEL_PATH = r"C:\Local LLM and Embedding Models\Local LLM\mistral-7b-instruct-gguf\mistral-7b-instruct-v0.1.Q5_K_M.gguf"


In [ ]:
# ------------------------
# 1. Load Local Mistral LLM (GGUF via ctransformers)
# ------------------------
llm = CTransformers(
    model=MODEL_PATH,
    model_type="mistral",   # important for tokenizer handling
    max_new_tokens=200,
    temperature=0.7,
    gpu_layers=0            # set >0 if you have GPU build of llama.cpp
)

In [ ]:
# ------------------------
# 2. Embeddings (Local MiniLM)
# ------------------------
embedding_model = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL_PATH)

In [ ]:
# ------------------------
# 3. Document Loader
# ------------------------
def document_loader(file):
    loader = PyPDFLoader(file.name)
    docs = loader.load()
    return docs

In [ ]:
# ------------------------
# 4. Text Splitter
# ------------------------
def text_splitter(docs):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=50,
        length_function=len,
    )
    return splitter.split_documents(docs)


In [ ]:
# ------------------------
# 5. Vector DB
# ------------------------
def build_vector_db(chunks):
    vectordb = Chroma.from_documents(chunks, embedding_model, persist_directory=CHROMA_DIR)
    vectordb.persist()
    return vectordb

In [ ]:
# ------------------------
# 6. RAG QA Function
# ------------------------
def rag_qa(file, query):
    docs = document_loader(file)
    chunks = text_splitter(docs)
    vectordb = build_vector_db(chunks)

    retriever = vectordb.as_retriever(search_kwargs={"k": 2})

    qa = RetrievalQA.from_chain_type(
        llm=llm,
        chain_type="stuff",
        retriever=retriever,
        return_source_documents=False
    )

    result = qa.run(query)
    return result

In [ ]:
# ------------------------
# 7. Gradio UI
# ------------------------
rag_app = gr.Interface(
    fn=rag_qa,
    inputs=[
        gr.File(label="Upload PDF", file_types=[".pdf"]),
        gr.Textbox(label="Your Question", lines=2, placeholder="Ask a question...")
    ],
    outputs=gr.Textbox(label="Answer"),
    title="Local Mistral RAG Bot",
    description="Fully offline PDF Q&A system using Mistral-7B-Instruct (GGUF) + MiniLM embeddings"
)

if __name__ == "__main__":
    rag_app.launch(server_name="127.0.0.1", server_port=7860)